# The recurring dark families of the cryptic plasmidome

The dark proteome of the payload-free small plasmids is **378,552 proteins that PlasAnn could
only call `ORF`**. `dark_plasmidome.ipynb` established that it is recurrent rather than
idiosyncratic, that the recurrence is not clonal redundancy, and that most of it survives
Pfam-A. This notebook asks the next question: **which** families recur, what the single most
prevalent one is, and whether the problem can be narrowed to a set small enough to work on.

Three questions, in order:

1. Do dark ORFs actually repeat across plasmids, or is every one a one-off?
2. How concentrated is the recurrence — is there a core?
3. What is the most prevalent family, and what does a tractable target set look like?

Everything traces to `scripts/cluster_dark_orfs.sh` (MMseqs2 `easy-cluster`, 30% identity /
80% bidirectional coverage) and `scripts/pfam_array.sbatch` (`hmmsearch --cut_ga` against
Pfam-A 38.2). Method and limits: `reports/dark_orf_clustering.md`,
`reports/pfam_dark_validation.md`.

In [ ]:
# The recurring dark protein families of the cryptic plasmidome.
# Inputs, all produced earlier and documented in reports/dark_orf_clustering.md and
# reports/pfam_dark_validation.md:
#   scripts/cluster_dark_orfs.sh          -> dark30_cluster.tsv, darkfam_stats.tsv
#   scripts/summarize_dark_orf_clusters.py -> darkfam_stats.tsv columns
#   scripts/pfam_array.sbatch + summarize_pfam.py -> darkfam_pfam.tsv
# Clustering is MMseqs2 easy-cluster at 30% identity / 80% bidirectional coverage; a "family"
# is one cluster, named by its representative sequence ID (PLASMID|cds_index).
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = Path.cwd()
while not (ROOT / "data" / "plasmidscope_primary").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
DOR, PFM, PP = (ROOT / "data" / d for d in ("dark_orf_run", "pfam_run", "plasmidscope_primary"))

# Palette: slots 1-4 of the validated categorical default + a neutral. Grey is reserved for
# absence, and carries exactly one meaning inside any single figure.
S1, S2, S3, S4, NEUT = "#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#b8b7ae"
INK, INK2, GRID, FAINT = "#0b0b0b", "#52514e", "#dcdbd4", "#d4d3cc"


def tidy(ax):
    for s in ("top", "right"):
        ax.spines[s].set_visible(False)
    for s in ("left", "bottom"):
        ax.spines[s].set(color=GRID, linewidth=1)
    ax.tick_params(colors=INK2, labelsize=9, length=0)
    ax.set_axisbelow(True)


FAM = pd.read_csv(DOR / "darkfam_stats.tsv", sep="\t").merge(
    pd.read_csv(PFM / "darkfam_pfam.tsv", sep="\t")[["rep", "pfam_hit", "pfam_acc", "pfam_name"]],
    on="rep", how="left")
FAM["pfam_hit"] = FAM["pfam_hit"].fillna(False)

CL = pd.read_csv(DOR / "dark30_cluster.tsv", sep="\t", header=None, names=["rep", "member"])
CL["plasmid_id"] = CL["member"].str.rsplit("|", n=1).str[0]

MET = pd.read_csv(PP / "plasmid_metadata_master.tsv", sep="\t", low_memory=False).merge(
    pd.read_csv(PP / "working_set.tsv", sep="\t", low_memory=False,
                usecols=["plasmid_id", "organism"]), on="plasmid_id", how="left")

NORF, NFAM = len(CL), len(FAM)
CARRIERS = CL["plasmid_id"].nunique()
assert NORF == 378_552 and NFAM == 92_752, (NORF, NFAM)
assert FAM["proteins"].sum() == NORF
# The subset was built from the cryptic small plasmids, which already exclude the two artifact
# habitat buckets; re-assert it rather than trust it.
assert MET.set_index("plasmid_id").loc[list(CL["plasmid_id"].unique()), "hab_top"].isin(
    ["Simulated-artifact", "Lab-artifact"]).sum() == 0

print(f"dark ORFs {NORF:,}  families {NFAM:,}  carrier plasmids {CARRIERS:,}")
print(f"Pfam-A 38.2 at --cut_ga names {FAM['pfam_hit'].sum():,} families "
      f"({FAM['pfam_hit'].mean() * 100:.1f}%), covering "
      f"{FAM.loc[FAM['pfam_hit'], 'proteins'].sum() / NORF * 100:.1f}% of dark ORFs")

## 1. Do dark ORFs repeat across plasmids?

The unit matters. A family counted once per *protein* and a family counted once per *plasmid*
give different answers, and so does asking "how many families" versus "how many ORFs".

In [ ]:
# Q1 — do dark ORFs repeat across plasmids at all, or is every one a one-off?
# The unit is the plasmid, not the protein: a family that appears twice on the same replicon has
# not travelled. FAM["plasmids"] is already the distinct-plasmid count per family.
BANDS = [(1, 1), (2, 4), (5, 9), (10, 49), (50, 99), (100, 499), (500, 10 ** 9)]
LABS = ["1", "2-4", "5-9", "10-49", "50-99", "100-499", "500+"]
rows = []
for (lo, hi), lab in zip(BANDS, LABS):
    s = FAM[FAM["plasmids"].between(lo, hi)]
    rows.append({"band": lab, "families": len(s), "% families": len(s) / NFAM * 100,
                 "dark ORFs": s["proteins"].sum(),
                 "% dark ORFs": s["proteins"].sum() / NORF * 100})
REC = pd.DataFrame(rows).set_index("band")
print("families by how many distinct plasmids they occur on:")
print(REC.round(1).to_string())

one = FAM["plasmids"].eq(1)
print(f"\nseen on exactly one plasmid: {one.sum():,} families ({one.mean() * 100:.1f}%) "
      f"holding {FAM.loc[one, 'proteins'].sum():,} dark ORFs "
      f"({FAM.loc[one, 'proteins'].sum() / NORF * 100:.1f}%)")
print(f"recurrent (>= 2 plasmids):   {(~one).sum():,} families ({(~one).mean() * 100:.1f}%) "
      f"holding {FAM.loc[~one, 'proteins'].sum():,} dark ORFs "
      f"({FAM.loc[~one, 'proteins'].sum() / NORF * 100:.1f}%)")

# The answer flips depending on which unit you count, so both are stated. Most families are
# one-offs; most dark ORFs sit in families that are not.
print("\nso: counted as families the dark proteome looks idiosyncratic, counted as ORFs it does "
      "not.\nThe rest of this notebook follows the ORFs.")

In [ ]:
# Figure 1. Grey = the non-recurrent band (one plasmid only); it means "does not travel" and
# nothing else in this figure.
fig, (axA, axB) = plt.subplots(1, 2, figsize=(13, 4.4), gridspec_kw={"wspace": 0.22})
x = np.arange(len(REC))
for ax, col, ttl in ((axA, "families", "families"), (axB, "dark ORFs", "dark ORFs")):
    v = REC[col].values
    ax.bar(x, v, width=0.56, color=[NEUT] + [S1] * (len(REC) - 1))
    for i, val in enumerate(v):
        ax.text(i, val + v.max() * 0.02, f"{val:,}\n{val / v.sum() * 100:.0f}%",
                ha="center", fontsize=8.5, color=INK2)
    ax.set(xticks=x, xticklabels=REC.index, yticks=[], ylim=(0, v.max() * 1.2))
    ax.set_xlabel("distinct plasmids the family occurs on", fontsize=9, color=INK2)
    tidy(ax)
axA.set_title(f"A — most families are one-offs\n{int(REC.loc['1', 'families']):,} of {NFAM:,} "
              f"({REC.loc['1', '% families']:.0f}%) sit on a single plasmid",
              fontsize=10.5, color=INK, loc="left")
axB.set_title(f"B — but most dark ORFs are not\n"
              f"{100 - REC.loc['1', '% dark ORFs']:.0f}% of the {NORF:,} dark ORFs are in "
              f"families that recur", fontsize=10.5, color=INK, loc="left")
fig.suptitle("Do dark ORFs repeat across plasmids?  Yes — and the answer depends on what you count",
             fontsize=12, color=INK, x=0.006, ha="left", y=1.02)
plt.show()

## 2. How concentrated is the recurrence?

If a few dozen families held most of the dark proteome, this would be a small problem. The
shape of the curve decides whether "narrowing" is possible at all — and what it costs.

In [ ]:
# Q2 — how concentrated is the recurrence? If a handful of families carried most of the dark
# proteome, the "dark matter" problem would be small and tractable. They do not, quite.
O = FAM.sort_values("proteins", ascending=False).reset_index(drop=True)
O["cum"] = O["proteins"].cumsum() / NORF
print("families needed to account for a given share of the dark proteome:")
for t in (0.10, 0.25, 0.50, 0.80, 0.90):
    n = int((O["cum"] < t).sum()) + 1
    print(f"  {t * 100:>3.0f}%  ->  top {n:>6,} families  ({n / NFAM * 100:5.2f}% of families)")

print("\nthresholds on plasmid spread — the candidate 'core' definitions:")
for k in (10, 50, 100, 200, 500, 1000):
    s = FAM[FAM["plasmids"] >= k]
    print(f"  on >= {k:>4} plasmids: {len(s):>6,} families  {s['proteins'].sum():>7,} dark ORFs "
          f"({s['proteins'].sum() / NORF * 100:5.1f}%)  "
          f"Pfam-named {s['pfam_hit'].mean() * 100:4.0f}%")

# Pfam status is not independent of prevalence: the commoner a family is, the likelier Pfam has
# a name for it. That is the shape you expect if prevalence and characterisation go together,
# and it is why "most prevalent" and "most interesting" are not the same question.
print("\nPfam-named share by prevalence band:")
print(pd.DataFrame({
    "families": FAM.groupby(pd.cut(FAM["plasmids"], [0, 1, 4, 9, 49, 99, 499, 10 ** 9],
                                   labels=LABS), observed=True).size(),
    "% Pfam-named": FAM.groupby(pd.cut(FAM["plasmids"], [0, 1, 4, 9, 49, 99, 499, 10 ** 9],
                                       labels=LABS), observed=True)["pfam_hit"].mean() * 100,
}).round(1).to_string())

In [ ]:
# Figure 2. One series per panel, so no colour semantics to collide; the reference lines are the
# thresholds quoted in the text above.
fig, (axA, axB) = plt.subplots(1, 2, figsize=(13, 4.4), gridspec_kw={"wspace": 0.24})

rank = np.arange(1, NFAM + 1)
axA.plot(rank, O["cum"].values * 100, color=S1, linewidth=2)
for t, lab in ((0.25, "25%"), (0.50, "50%")):
    n = int((O["cum"] < t).sum()) + 1
    axA.plot([n, n], [0, t * 100], color=GRID, linewidth=1, zorder=0)
    axA.plot([1, n], [t * 100, t * 100], color=GRID, linewidth=1, zorder=0)
    axA.scatter([n], [t * 100], s=26, color=INK, zorder=3)
    axA.text(n * 1.25, t * 100 - 4, f"{lab} of dark ORFs\nin top {n:,} families",
             fontsize=8.5, color=INK2)
axA.set(xscale="log", xlim=(1, NFAM), ylim=(0, 100), yticks=[0, 25, 50, 75, 100])
axA.set_xlabel("families, ranked by size (log)", fontsize=9, color=INK2)
axA.set_ylabel("cumulative % of dark ORFs", fontsize=9, color=INK2)
axA.set_title(f"A — no small core explains the dark proteome\n"
              f"half of it needs {int((O['cum'] < 0.5).sum()) + 1:,} of the {NFAM:,} families",
              fontsize=10.5, color=INK, loc="left")
tidy(axA)

ks = [10, 50, 100, 200, 500, 1000]
cov = [FAM.loc[FAM["plasmids"] >= k, "proteins"].sum() / NORF * 100 for k in ks]
nfa = [int((FAM["plasmids"] >= k).sum()) for k in ks]
xb = np.arange(len(ks))
axB.bar(xb, cov, width=0.5, color=S1)
for i, (c, n) in enumerate(zip(cov, nfa)):
    axB.text(i, c + 1.2, f"{c:.1f}%\n{n:,} fam", ha="center", fontsize=8.5, color=INK2)
axB.set(xticks=xb, xticklabels=[f">={k}" for k in ks], yticks=[],
        ylim=(0, max(cov) * 1.28))
axB.set_xlabel("family occurs on at least this many plasmids", fontsize=9, color=INK2)
axB.set_title("B — what each cut-off buys you\nshare of the dark proteome retained",
              fontsize=10.5, color=INK, loc="left")
tidy(axB)

fig.suptitle("How concentrated is the recurrence?", fontsize=12, color=INK, x=0.006,
             ha="left", y=1.02)
plt.show()

## 3. The most prevalent dark family

Ranked by distinct plasmids rather than protein count, because the question is how widely the
family travels, not how many copies exist.

In [ ]:
# Q3 — the single most prevalent dark family. Ranked by distinct plasmids, not by protein count,
# because the question is how widely it travels.
TOP = FAM.sort_values("plasmids", ascending=False).iloc[0]
tmem = CL[CL["rep"].eq(TOP["rep"])]
tpl = MET[MET["plasmid_id"].isin(set(tmem["plasmid_id"]))]
BG = MET[MET["plasmid_id"].isin(set(CL["plasmid_id"]))]          # all dark-ORF carriers

print(f"most prevalent dark family: {TOP['rep']}")
print(f"  {TOP['proteins']:,} proteins on {TOP['plasmids']:,} distinct plasmids "
      f"({TOP['plasmids'] / CARRIERS * 100:.1f}% of the {CARRIERS:,} carriers)")
print(f"  spans {TOP['lineages']} MOB primary clusters and {TOP['habitats']} habitats")
print(f"  representative is {TOP['rep_len_aa']} aa")
print(f"  Pfam-A 38.2 at --cut_ga: {'HIT ' + str(TOP['pfam_name']) if TOP['pfam_hit'] else 'NO HIT'}")

print(f"\ncarrier plasmids (n = {len(tpl):,}) vs all dark-ORF carriers (n = {len(BG):,}):")
print(f"  size_bp   median {tpl['size_bp'].median():,.0f} (IQR {tpl['size_bp'].quantile(.25):,.0f}"
      f"-{tpl['size_bp'].quantile(.75):,.0f})   background median {BG['size_bp'].median():,.0f}")
print(f"  mobility  {tpl['mob_mobility'].value_counts(normalize=True).mul(100).round(0).head(3).to_dict()}"
      f"   background {BG['mob_mobility'].value_counts(normalize=True).mul(100).round(0).head(3).to_dict()}")
print(f"  replicon  {tpl['mob_rep_types'].isna().mean() * 100:.0f}% have no mob_typer call "
      f"(background {BG['mob_rep_types'].isna().mean() * 100:.0f}%)")
print(f"  organism  named for only {tpl['organism'].notna().sum():,} of {len(tpl):,} "
      f"({tpl['organism'].notna().mean() * 100:.1f}%) — these are mostly IMG/PR metagenomic "
      f"plasmids with no isolate host")

# Habitat, against the background of all dark-ORF carriers. Enrichment, not raw share, is the
# claim: the background is itself 20% human.
hb = tpl["hab_sub"].value_counts(normalize=True) * 100
bb = BG["hab_sub"].value_counts(normalize=True) * 100
HB = pd.DataFrame({"family %": hb, "background %": bb.reindex(hb.index)}).head(8)
HB["fold"] = HB["family %"] / HB["background %"]
print("\nhabitat of the carriers (top 8 for this family):")
print(HB.round(1).to_string())

# Lineage spread. "88 clusters" sounds broad; the concentration inside those clusters is what
# decides whether this is one clone sequenced repeatedly.
vc = tpl["mob_cluster"].value_counts()
print(f"\nMOB lineage spread: {tpl['mob_cluster'].nunique()} primary clusters over "
      f"{tpl['mob_cluster'].notna().sum():,} typed plasmids | "
      f"largest {vc.iloc[0] / vc.sum() * 100:.0f}% | top 3 {vc.head(3).sum() / vc.sum() * 100:.0f}%")

In [ ]:
# Figure 3 — portrait of the most prevalent family. Grey means "all dark-ORF carriers"
# (the background) in every panel; slot 1 is this family. That is the only contrast drawn here.
fig, axes = plt.subplots(1, 4, figsize=(16, 4.2), gridspec_kw={"wspace": 0.3})
axA, axB, axC, axD = axes

hh = HB.head(6).iloc[::-1]
y = np.arange(len(hh))
axA.barh(y + 0.19, hh["background %"], height=0.34, color=NEUT)
axA.barh(y - 0.19, hh["family %"], height=0.34, color=S1)
for i, (f, b) in enumerate(zip(hh["family %"], hh["background %"])):
    axA.text(max(f, b) + 2, i, f"{f / b:.1f}x", va="center", fontsize=8.5, color=INK2)
axA.set(yticks=y, yticklabels=[s[:22] for s in hh.index], xticks=[],
        xlim=(0, hh[["family %", "background %"]].values.max() * 1.28))
axA.set_xlabel("% of carrier plasmids", fontsize=9, color=INK2)
axA.set_title("A — habitat\nblue = family, grey = background", fontsize=10, color=INK, loc="left")
tidy(axA)

for lab, d, col in (("background", BG["size_bp"], NEUT), ("family", tpl["size_bp"], S1)):
    axB.hist(d, bins=60, range=(0, 20_000), density=True, histtype="stepfilled",
             color=col, alpha=0.55 if col == NEUT else 0.85, linewidth=0)
axB.axvline(tpl["size_bp"].median(), color=INK, linewidth=1.4)
axB.text(tpl["size_bp"].median() + 400, axB.get_ylim()[1] * 0.9,
         f"median\n{tpl['size_bp'].median():,.0f} bp", fontsize=8.5, color=INK, va="top")
axB.set(xlim=(0, 20_000), yticks=[], xticks=[0, 5_000, 10_000, 15_000, 20_000])
axB.xaxis.set_major_formatter(lambda v, _: f"{v:,.0f}")
axB.set_xlabel("carrier plasmid size (bp)", fontsize=9, color=INK2)
axB.set_title("B — carrier size\ndensity, so the two are comparable", fontsize=10, color=INK,
              loc="left")
tidy(axB)

# C — lineage concentration. A family on many clusters can still be one clone; the curve says
# how fast the carriers pile into their biggest clusters.
for lab, s, col in (("background", BG["mob_cluster"], NEUT), ("family", tpl["mob_cluster"], S1)):
    v = s.value_counts().values
    axC.plot(np.arange(1, len(v) + 1), np.cumsum(v) / v.sum() * 100, color=col, linewidth=2)
axC.text(2, 92, "this family", fontsize=9, color=S1)
axC.text(12, 42, "background", fontsize=9, color=INK2)
axC.set(xscale="log", ylim=(0, 100), yticks=[0, 25, 50, 75, 100])
axC.set_xlabel("MOB primary clusters, ranked (log)", fontsize=9, color=INK2)
axC.set_ylabel("cumulative % of carriers", fontsize=9, color=INK2)
axC.set_title(f"C — lineage spread\n{tpl['mob_cluster'].nunique()} clusters, but top 3 hold "
              f"{tpl['mob_cluster'].value_counts().head(3).sum() / tpl['mob_cluster'].notna().sum() * 100:.0f}%",
              fontsize=10, color=INK, loc="left")
tidy(axC)

rt = pd.DataFrame({
    "family": [tpl["mob_rep_types"].isna().mean() * 100, tpl["mob_rep_types"].notna().mean() * 100],
    "background": [BG["mob_rep_types"].isna().mean() * 100, BG["mob_rep_types"].notna().mean() * 100],
}, index=["no call", "typed"])
yb = np.arange(2)
axD.barh(yb + 0.19, rt["background"], height=0.34, color=NEUT)
axD.barh(yb - 0.19, rt["family"], height=0.34, color=S1)
for i in yb:
    axD.text(max(rt.iloc[i]) + 2, i, f"{rt['family'].iloc[i]:.0f}% vs {rt['background'].iloc[i]:.0f}%",
             va="center", fontsize=8.5, color=INK2)
axD.set(yticks=yb, yticklabels=rt.index, xticks=[], xlim=(0, 118))
axD.set_xlabel("% of carrier plasmids", fontsize=9, color=INK2)
axD.set_title("D — replicon typing\nmob_typer mob_rep_types only", fontsize=10, color=INK,
              loc="left")
tidy(axD)

fig.suptitle(f"The most prevalent dark family — {TOP['rep']}\n"
             f"{TOP['proteins']:,} proteins on {TOP['plasmids']:,} plasmids "
             f"({TOP['plasmids'] / CARRIERS * 100:.1f}% of carriers), {TOP['rep_len_aa']} aa, "
             f"no Pfam-A hit at the curated gathering threshold",
             fontsize=12, color=INK, x=0.006, ha="left", y=1.06)
plt.show()

## 4. The top 25 — and which of them are actually unexplained

Prevalence and darkness are not the same property. The Pfam column splits this list in two,
and the split is the point.

In [ ]:
# Q4 — what are the other prevalent families? The Pfam column is doing the work here: it splits
# the top of the list into "PlasAnn should have called this" and "nobody can name it".
N = 25
T = FAM.sort_values("plasmids", ascending=False).head(N).copy()
T["dom_hab"] = [CL[CL["rep"].eq(r)].drop_duplicates("plasmid_id")
                .merge(MET[["plasmid_id", "hab_sub"]], on="plasmid_id")["hab_sub"]
                .value_counts(normalize=True).pipe(lambda s: f"{s.index[0][:20]} {s.iloc[0]*100:.0f}%")
                for r in T["rep"]]
T["pfam"] = np.where(T["pfam_hit"], T["pfam_name"], "—")
print(f"top {N} dark families by distinct plasmids:")
print(T[["rep", "proteins", "plasmids", "lineages", "habitats", "rep_len_aa", "pfam",
         "dom_hab"]].to_string(index=False))

named = T[T["pfam_hit"]]
print(f"\n{len(named)} of the top {N} have a Pfam name: "
      f"{', '.join(sorted(named['pfam_name'].astype(str).unique()))}")
print("Read that list: replication, mobilisation, toxin-antitoxin, partition, resolvase. These are\n"
      "the plasmid backbone — PlasAnn failing to call them is an annotation gap, not new biology.\n"
      "(reports/pfam_dark_validation.md quantifies that gap at 18.6% of the dark proteome.)")

unnamed = T[~T["pfam_hit"]]
print(f"\n{len(unnamed)} of the top {N} have no Pfam name — these are the real objects of study:")
print(unnamed[["rep", "proteins", "plasmids", "lineages", "habitats", "rep_len_aa",
               "dom_hab"]].to_string(index=False))
print("\nThey split by lineage spread, and the split matters: a family on 800 plasmids but only\n"
      "3-9 MOB clusters is one lineage sequenced repeatedly, not a widely-distributed protein.")

In [ ]:
# Figure 4. Grey = the family already has a Pfam name, i.e. it is explained; slot 1 = still
# unnamed. Panel B re-plots the same families on lineage spread, which is what separates a
# genuinely mobile family from a well-sequenced clone.
fig, (axA, axB) = plt.subplots(1, 2, figsize=(15, 7.2),
                               gridspec_kw={"wspace": 0.05, "width_ratios": [1.35, 1]})
t = T.iloc[::-1]
y = np.arange(len(t))
cols = [NEUT if h else S1 for h in t["pfam_hit"]]

axA.barh(y, t["plasmids"], height=0.6, color=cols)
for i, (v, p, n) in enumerate(zip(t["plasmids"], t["pfam"], t["rep_len_aa"])):
    axA.text(v + T["plasmids"].max() * 0.012, i, f"{v:,}   {p}  ·  {n} aa",
             va="center", fontsize=8.5, color=INK2)
axA.set(yticks=y, yticklabels=[r.replace("IMGPR_plasmid_", "IMGPR_")[:34] for r in t["rep"]],
        xticks=[], xlim=(0, T["plasmids"].max() * 1.42))
axA.tick_params(axis="y", labelsize=7.5)
axA.set_xlabel("distinct plasmids carrying the family", fontsize=9, color=INK2)
axA.set_title(f"A — the {N} most prevalent dark families\n"
              f"grey: {int(T['pfam_hit'].sum())} have a Pfam name and are mostly plasmid "
              f"backbone · blue: {int((~T['pfam_hit']).sum())} have none",
              fontsize=10.5, color=INK, loc="left")
tidy(axA)

axB.scatter(t["lineages"], y, s=64, color=cols, zorder=3)
for i, v in enumerate(t["lineages"]):
    axB.text(v + 6, i, f"{v}", va="center", fontsize=8, color=INK2)
axB.axvline(10, color=GRID, linewidth=1, zorder=0)
axB.text(10.8, len(t) - 0.4, "10 lineages", fontsize=8.5, color=INK2)
axB.set(yticks=y, yticklabels=[], xlim=(0, t["lineages"].max() * 1.18), ylim=axA.get_ylim())
axB.set_xlabel("MOB primary clusters spanned", fontsize=9, color=INK2)
axB.set_title("B — is it mobile, or one clone?\nsame families, same colours",
              fontsize=10.5, color=INK, loc="left")
tidy(axB)

fig.suptitle("The prevalent dark families, and which of them are actually unexplained",
             fontsize=12, color=INK, x=0.006, ha="left", y=0.965)
plt.show()

## 5. Narrowing to a workable target set

Three conditions have to hold at once for a family to be worth pursuing: **recurrent** (not a
sequencing one-off), **lineage-spread** (not one clone sequenced repeatedly), and
**Pfam-negative** (something left to find). Each cut is stated and its cost printed.

In [ ]:
# Q5 — narrowing. A target set has to satisfy three things at once: recurrent (so it is not a
# sequencing one-off), lineage-spread (so it is not one clone), and Pfam-negative (so there is
# something left to find). Each cut is stated, and the cost of each is printed.
CUTS = [(50, 5), (100, 5), (100, 10), (100, 20), (200, 10)]
print("candidate core definitions (Pfam-negative in every row):")
for np_, nl in CUTS:
    c = FAM[(~FAM["pfam_hit"]) & (FAM["plasmids"] >= np_) & (FAM["lineages"] >= nl)]
    print(f"  >= {np_:>3} plasmids & >= {nl:>2} lineages : {len(c):>4} families, "
          f"{c['proteins'].sum():>6,} dark ORFs ({c['proteins'].sum() / NORF * 100:4.1f}% of dark), "
          f"on {c['plasmids'].sum():>6,} plasmid-slots")

MIN_PLASMIDS, MIN_LINEAGES = 100, 10
CORE = (FAM[(~FAM["pfam_hit"]) & (FAM["plasmids"] >= MIN_PLASMIDS)
            & (FAM["lineages"] >= MIN_LINEAGES)]
        .sort_values("plasmids", ascending=False).reset_index(drop=True))
core_pl = CL[CL["rep"].isin(set(CORE["rep"]))]["plasmid_id"].nunique()

print(f"\nCORE = Pfam-negative, >= {MIN_PLASMIDS} plasmids, >= {MIN_LINEAGES} MOB lineages")
print(f"  {len(CORE):,} families")
print(f"  {CORE['proteins'].sum():,} dark ORFs ({CORE['proteins'].sum() / NORF * 100:.1f}% of the "
      f"dark proteome)")
print(f"  touch {core_pl:,} distinct plasmids ({core_pl / CARRIERS * 100:.1f}% of the "
      f"{CARRIERS:,} carriers)")
print(f"  length: median {CORE['rep_len_aa'].median():.0f} aa, IQR "
      f"{CORE['rep_len_aa'].quantile(.25):.0f}-{CORE['rep_len_aa'].quantile(.75):.0f}, "
      f"max {CORE['rep_len_aa'].max():.0f}")
print(f"  lineages: median {CORE['lineages'].median():.0f}, max {CORE['lineages'].max()}")
print(f"  habitats: median {CORE['habitats'].median():.0f}, max {CORE['habitats'].max()}")

out = ROOT / "data" / "dark_orf_run" / "core_dark_families.tsv"
CORE.to_csv(out, sep="\t", index=False)
print(f"\nwritten -> {out.relative_to(ROOT)}  ({len(CORE)} rows)")
print("\nThis is the set worth structure prediction and targeted work: small enough to fold,\n"
      "large enough to matter, and every member has already survived PlasAnn, a named-protein\n"
      "co-clustering check, and Pfam-A at the curated gathering threshold.")
print(f"\ntop 15 of the core:")
print(CORE.head(15)[["rep", "proteins", "plasmids", "lineages", "habitats",
                     "rep_len_aa"]].to_string(index=False))

In [ ]:
# Figure 5. The two cuts are drawn as lines so the choice is visible rather than asserted.
# Grey = a family Pfam can name (excluded from the core); slot 1 = Pfam-negative (eligible).
sel = FAM[FAM["plasmids"] >= 20]
fig = plt.figure(figsize=(14, 5.2))
gs = fig.add_gridspec(1, 2, width_ratios=[1.35, 1], wspace=0.24)
axA, axB = fig.add_subplot(gs[0, 0]), fig.add_subplot(gs[0, 1])

for hit, col, a, z in ((True, NEUT, 0.55, 1), (False, S1, 0.8, 2)):
    s = sel[sel["pfam_hit"].eq(hit)]
    axA.scatter(s["plasmids"], s["lineages"], s=13, color=col, alpha=a, linewidth=0, zorder=z)
axA.axvline(MIN_PLASMIDS, color=INK, linewidth=1.2, zorder=3)
axA.axhline(MIN_LINEAGES, color=INK, linewidth=1.2, zorder=3)
axA.text(MIN_PLASMIDS * 1.1, sel["lineages"].max() * 0.72,
         f"CORE\n{len(CORE)} families\n{CORE['proteins'].sum():,} dark ORFs",
         fontsize=9.5, color=INK)
axA.set(xscale="log", yscale="log", xlim=(20, sel["plasmids"].max() * 1.3),
        ylim=(1, sel["lineages"].max() * 1.4))
axA.set_xlabel("distinct plasmids (log)", fontsize=9, color=INK2)
axA.set_ylabel("MOB primary clusters spanned (log)", fontsize=9, color=INK2)
axA.set_title(f"A — the two cuts, on the {len(sel):,} families seen on >= 20 plasmids\n"
              f"grey = Pfam can name it, so it leaves the core",
              fontsize=10.5, color=INK, loc="left")
tidy(axA)

steps = ["all dark ORFs", "in families\non >=100 plasmids", "and >=10\nlineages",
         "and Pfam-\nnegative"]
vals = [NORF,
        FAM.loc[FAM["plasmids"] >= MIN_PLASMIDS, "proteins"].sum(),
        FAM.loc[(FAM["plasmids"] >= MIN_PLASMIDS) & (FAM["lineages"] >= MIN_LINEAGES),
                "proteins"].sum(),
        CORE["proteins"].sum()]
xb = np.arange(len(vals))
# All four bars are the same series (dark ORFs retained), so they take one colour:
# grey stays reserved for "Pfam can name it" from panel A.
axB.bar(xb, vals, width=0.5, color=S1)
for i, v in enumerate(vals):
    axB.text(i, v + NORF * 0.02, f"{v:,}\n{v / NORF * 100:.1f}%", ha="center", fontsize=8.5,
             color=INK2)
axB.set(xticks=xb, xticklabels=steps, yticks=[], ylim=(0, NORF * 1.2))
axB.tick_params(axis="x", labelsize=8.5)
axB.set_title("B — what each cut costs\ndark ORFs retained", fontsize=10.5, color=INK, loc="left")
tidy(axB)

fig.suptitle(f"Narrowing to a workable target set — {len(CORE)} families, "
             f"{CORE['proteins'].sum() / NORF * 100:.1f}% of the dark proteome",
             fontsize=12, color=INK, x=0.006, ha="left", y=1.0)
plt.show()

## 6. How much of this depends on the clustering threshold?

Sections 1–5 all rest on one clustering. Two things were wrong with treating that as settled.
The threshold was never varied, and MMseqs2's cascaded clustering **was not enforcing the
criterion the methods claim** — `--cluster-reassign` was not set, and an audit
(`scripts/audit_cluster_compliance.py`) found only **89.1%** of sampled members met 30% identity
and 80% coverage against their own representative.

The clustering was therefore rebuilt at four identity thresholds, all with reassignment on and
coverage held at 80% both ways. The thresholds are literature landmarks rather than arbitrary:
**50% = UniRef50**, **70% = NMPFamsDB** (*NAR* 2024), **90% = UniRef90**. Sweeping rather than
picking one follows PlasX/MobMess (*Nat Microbiol* 2024), which ran twelve on plasmid gene
families.

Full method, literature table and compliance audit: `reports/clustering_threshold_sensitivity.md`.

In [ ]:
# Threshold sensitivity. Everything above is conditional on ONE clustering: 30% identity,
# 80% bidirectional coverage, and — as it turned out — without --cluster-reassign, so ~11% of
# members did not actually meet that criterion. scripts/recluster_sweep.sbatch rebuilds the
# clustering at 30/50/70/90% WITH reassignment; full method and literature comparison in
# reports/clustering_threshold_sensitivity.md.
RC = ROOT / "data" / "dark_orf_run" / "recluster"
LEVELS = [("30% + reassign", "dark30"), ("50% (UniRef50)", "dark50"),
          ("70% (NMPFamsDB)", "dark70"), ("90% (UniRef90)", "dark90")]


def headline(f):
    core = f[(~f["pfam_hit"]) & (f["plasmids"] >= 100) & (f["lineages"] >= 10)]
    top = f.sort_values("plasmids", ascending=False).iloc[0]
    return {"families": len(f),
            "% ORFs recurrent": f.loc[f["plasmids"] >= 2, "proteins"].sum() / NORF * 100,
            "% ORFs Pfam-named": f.loc[f["pfam_hit"], "proteins"].sum() / NORF * 100,
            "core families": len(core), "core % of ORFs": core["proteins"].sum() / NORF * 100,
            "top fam plasmids": int(top["plasmids"]), "top fam lineages": int(top["lineages"])}


SENS = pd.DataFrame(
    [headline(FAM.assign(pfam_hit=FAM["pfam_hit"].fillna(False)))]
    + [headline(pd.read_csv(RC / f"{t}_fampfam.tsv", sep="\t")) for _, t in LEVELS],
    index=["30% (this notebook)"] + [lab for lab, _ in LEVELS])
print(SENS.round(1).to_string())

fig, axes = plt.subplots(1, 3, figsize=(14.5, 4), gridspec_kw={"wspace": 0.3})
x = np.arange(len(SENS))
for ax, col, ttl, sub in (
        (axes[0], "% ORFs recurrent", "Survives", "dark ORFs in families on >= 2 plasmids"),
        (axes[1], "% ORFs Pfam-named", "Survives", "dark ORFs in Pfam-named families"),
        (axes[2], "core families", "Does NOT survive", "Pfam-negative, >=100 plasmids, >=10 lineages")):
    v = SENS[col].values
    ax.bar(x, v, width=0.55, color=[NEUT] + [S1] * (len(SENS) - 1))
    for i, val in enumerate(v):
        ax.text(i, val + v.max() * 0.03, f"{val:,.0f}" if col == "core families" else f"{val:.0f}%",
                ha="center", fontsize=9, color=INK2)
    ax.set(xticks=x, xticklabels=["30%\norig", "30%\n+RA", "50%", "70%", "90%"], yticks=[],
           ylim=(0, v.max() * 1.22))
    ax.tick_params(axis="x", labelsize=8.5)
    ax.set_title(f"{ttl}\n{sub}", fontsize=10, color=INK, loc="left")
    tidy(ax)
fig.suptitle("Which results depend on the clustering threshold, and which do not\n"
             "grey = the 30% clustering this notebook uses", fontsize=12, color=INK,
             x=0.006, ha="left", y=1.06)
plt.show()

print(f"\nThe recurrence result and the Pfam verdict hold at every threshold, down to "
      f"{SENS['% ORFs recurrent'].min():.0f}% recurrent even at UniRef90.")
print(f"The core target set does not: {SENS.loc['30% (this notebook)', 'core families']:.0f} "
      f"families here, {SENS.loc['30% + reassign', 'core families']:.0f} once the criterion is "
      f"actually enforced at the same identity, {SENS.loc['50% (UniRef50)', 'core families']:.0f} "
      f"at UniRef50. Treat 93 as an upper bound.")

## What this says

**Dark ORFs repeat, heavily.** 63% of families sit on exactly one plasmid — but those one-offs
are only 16% of the dark proteome. **84% of dark ORFs are in families seen on two or more
plasmids**, and a quarter of the whole dark proteome sits in just 399 families. Section 6 shows
this survives every threshold tested, down to 70% of dark ORFs still recurrent at UniRef90.

**There is no small core.** Half of it still needs 3,009 families. The long tail is real, not
an artifact of where the clustering threshold was set.

**The most prevalent family has no name.** `IMGPR_plasmid_3300022589_000018|2` is on 2,468
plasmids — 3.9% of every plasmid here that carries a dark ORF — across 88 MOB primary clusters
and 17 habitats, and Pfam-A 38.2 has nothing for it at the curated gathering threshold. It is a
436 aa protein sitting on ~6 kb, non-mobilizable, human-associated plasmids that are 86%
untypeable by replicon (background 69%). **This one is threshold-dependent** — see below.

**Prevalence and darkness pull apart.** 17 of the 25 most prevalent families *do* have a Pfam
name, and the names are Rep_1, Rep3_C, Replicase, Mob_Pre, NikA-like, RelB, YoeB/YafQ_toxin,
wHTH_Par, TraD, Resolvase — the plasmid backbone. Those are PlasAnn annotation failures, not
discoveries. The gradient is systematic: Pfam names 24% of families seen on one plasmid and 66%
of those seen on 500+. **The commonest dark proteins are mostly just badly annotated ones.**

**The narrowed set is 93 families.** Pfam-negative, on >= 100 plasmids, spanning >= 10 MOB
lineages: 22,341 dark ORFs (5.9% of the dark proteome) touching 16,757 plasmids (26% of all
carriers), median 130 aa. Written to `data/dark_orf_run/core_dark_families.tsv`. Every member
has already survived PlasAnn, the named-protein co-clustering check, and Pfam-A at `--cut_ga`.
**Treat 93 as an upper bound** — section 6 puts it at 61 once the clustering criterion is
properly enforced at the same identity, and 39 at UniRef50.

### One caveat, specific to the top family

78% of its carriers are `Human: unspecified` (background 20%, so ~3.9x enriched), and its 88
MOB clusters are top-heavy — **the largest three hold 70% of carriers**. It is genuinely
widespread, but a large part of that spread is one well-sequenced human-associated lineage
group. `IMGPR_plasmid_3300042343_000004|14` (84 lineages) and `GenBank_CP027661.1|11` (76
lineages, 28 habitats, dominant habitat only 22%) are flatter across lineages and may be the
better targets despite being smaller.

Section 6 sharpens this further: at 50% identity the same ORF falls into a family of 89
proteins spanning 5 lineages, and at 70% into one of 5 proteins in a single lineage. The "88
lineages" figure is a property of the 30% threshold, not of a protein.

### What this does not establish

The family boundaries are one defensible choice among several — section 6 quantifies exactly
which conclusions move when that choice changes and which do not. And "no Pfam hit at
`--cut_ga`" means no *curated* match; it is not proof of structural novelty. That question needs
the structures, and then a structural search against the AlphaFold DB / PDB.